## Setup, Imports, and Triton Fixes

In [1]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
torchvision 0.26.0.dev20260315+cu128 requires torch==2.12.0.dev20260315, but you have torch 2.10.0 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.

In [2]:
import datasets
import trl
print("Successfully imported datasets version:", datasets.__version__)
print("Successfully imported trl version:", trl.__version__)

Successfully imported datasets version: 4.3.0
Successfully imported trl version: 0.24.0


In [3]:
import multiprocessing

multiprocessing.cpu_count()

48

In [ ]:
import os
import sys
import stat
import shutil
import gc
import zipfile
from pathlib import Path
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

def append_if_exists(path):
    if os.path.exists(path) and path not in sys.path:
        sys.path.append(path)
        print(f"append {path}")

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

append_if_exists("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages")
for p in Path("/kaggle/input").glob("**/python_packages"):
    append_if_exists(str(p))
for p in Path("/kaggle/input").glob("**/cutlass/python"):
    append_if_exists(str(p))

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst

print("Setup complete.")

## Data Loading & Formatting

In [ ]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

input_root = Path("/kaggle/input")
candidates = [
    input_root / "final-nemotron-training-data" / "final_Nemotron_training_data.csv",
    input_root / "final-nemotron-training-data" / "final_nemotron_training_data.csv",
]
for p in input_root.glob("**/final_Nemotron_training_data.csv"):
    candidates.append(p)
for p in input_root.glob("**/final_nemotron_training_data.csv"):
    candidates.append(p)

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(f"Could not find final_Nemotron_training_data.csv under {input_root}. Available inputs: {sorted(os.listdir(input_root))}")

print(f"Using training data: {data_path}")
train_df = pl.read_csv(str(data_path))
print(f"Total training samples available: {len(train_df)}")
train_df = train_df.sample(n=min(SAMPLE_SIZE, len(train_df)), seed=42)

hf_dataset = Dataset.from_pandas(train_df.to_pandas())

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_training_text(example, idx):
    prompt = example["prompt"]
    answer = example["answer"]
    cot = example["generated_cot"]

    user_msg = prompt + "\nPut your final answer inside \\boxed{}."
    if idx % 2 == 0:
        assistant_msg = cot if "\\boxed{" in cot else f"{cot}\n\n\\boxed{{{answer}}}"
    else:
        assistant_msg = f"\\boxed{{{answer}}}"

    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )
    return {"text": text}

In [ ]:
hf_dataset = hf_dataset.map(
    build_training_text,
    with_indices=True,
    remove_columns=hf_dataset.column_names,
)

print(f"Dataset formatted. Example:\n{hf_dataset[0]['text'][:500]}")

## Model Loading & LoRA Setup

In [ ]:
SAMPLE_SIZE = 800
LORA_RANK = 16
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 3
GRAD_ACCUM = 4
LR = 1e-4
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.cuda.empty_cache()
gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
print(f"Model loaded. Vocab size: {len(tokenizer)}")

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training with SFTTrainer

In [ ]:
import os
import triton.backends.nvidia.compiler as nv_compiler

os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args
)

print("Starting training...")
trainer.train()

## Save and Package Submission

In [ ]:
# Save adapter weights and config
trainer.model.save_pretrained(OUTPUT_DIR)
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

# Package into submission.zip (required by competition)
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)  # files at zip root

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

# Verify it contains adapter_config.json (required)
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("✓ submission.zip looks correct and is ready to submit!")